# DrishtiYOLO — Evaluation Notebook
mAP, per-class AP, confusion matrix, latency, and error analysis.

In [ ]:
# Cell 1 — Install + Drive mount
!nvidia-smi
!pip install -q ultralytics

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/drishti_runs'
BEST_PT     = f'{DRIVE_ROOT}/best_yolo11s_idd_v1_final.pt'
COCO_PT     = 'yolo11s.pt'
YAML_PATH   = '/content/idd.yaml'   # re-generate in Cell 2 if /content reset


In [ ]:
# Cell 2 — Rebuild dataset (run if /content was wiped between sessions)
# Re-extract IDD and regenerate YAML. Skip if /content/idd_yolo exists.
import os
from pathlib import Path
import yaml

YOLO_DATASET_DIR = Path('/content/idd_yolo')

if not YOLO_DATASET_DIR.exists():
    print('Dataset not found in /content — re-run cells 3-4 of drishti_train.ipynb first.')
else:
    dataset_cfg = {
        'path': str(YOLO_DATASET_DIR),
        'train': 'images/train',
        'val':   'images/val',
        'nc': 15,
        'names': [
            'auto_rickshaw', 'bicycle', 'bus', 'car', 'caravan',
            'electric_vehicle', 'motorcycle', 'person', 'rider',
            'traffic_light', 'traffic_sign', 'truck',
            'vehicle_fallback', 'animal', 'wheelchair'
        ]
    }
    with open(YAML_PATH, 'w') as f:
        yaml.dump(dataset_cfg, f)
    print(f'YAML written to {YAML_PATH}')


In [ ]:
# Cell 3 — mAP50 / mAP50-95 comparison table
from ultralytics import YOLO
import pandas as pd

def run_val(weights, yaml_path, label):
    model = YOLO(weights)
    metrics = model.val(data=yaml_path, imgsz=640, batch=16, verbose=False)
    return {
        'Model': label,
        'mAP50':    round(metrics.box.map50, 4),
        'mAP50-95': round(metrics.box.map,   4),
        'Precision':round(metrics.box.mp,    4),
        'Recall':   round(metrics.box.mr,    4),
    }

rows = [
    run_val(COCO_PT,  YAML_PATH, 'COCO baseline (yolo11s)'),
    run_val(BEST_PT,  YAML_PATH, 'DrishtiYOLO   (yolo11s)'),
]
df = pd.DataFrame(rows)
print(df.to_string(index=False))


In [ ]:
# Cell 4 — Per-class AP (headline result)
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

CLASSES = [
    'auto_rickshaw', 'bicycle', 'bus', 'car', 'caravan',
    'electric_vehicle', 'motorcycle', 'person', 'rider',
    'traffic_light', 'traffic_sign', 'truck',
    'vehicle_fallback', 'animal', 'wheelchair'
]
INDIA_SPECIFIC = {'auto_rickshaw', 'rider', 'vehicle_fallback', 'animal', 'caravan', 'electric_vehicle', 'wheelchair'}

def per_class_ap50(weights, yaml_path):
    m = YOLO(weights)
    val = m.val(data=yaml_path, imgsz=640, batch=16, verbose=False)
    return val.box.ap50  # shape: (nc,)

coco_ap = per_class_ap50(COCO_PT,  YAML_PATH)
drishti_ap = per_class_ap50(BEST_PT, YAML_PATH)

x = np.arange(len(CLASSES))
width = 0.35
colors = ['#e74c3c' if c in INDIA_SPECIFIC else '#3498db' for c in CLASSES]

fig, ax = plt.subplots(figsize=(16, 6))
bars_coco    = ax.bar(x - width/2, coco_ap,    width, label='COCO baseline', alpha=0.7, color='#95a5a6')
bars_drishti = ax.bar(x + width/2, drishti_ap, width, label='DrishtiYOLO',   alpha=0.9, color=colors)

ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('AP@50')
ax.set_title('Per-class AP50: COCO baseline vs DrishtiYOLO\n(red = India-specific classes)')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/drishti_runs/per_class_ap50.png', dpi=150)
plt.show()
print('Saved per_class_ap50.png to Drive.')


In [ ]:
# Cell 5 — Inference latency / FPS
import time, torch
from ultralytics import YOLO

def benchmark_latency(weights, n_warmup=20, n_runs=200):
    model = YOLO(weights)
    dummy = torch.zeros(1, 3, 640, 640).cuda()
    # warmup
    for _ in range(n_warmup):
        model.predict(source=dummy, verbose=False)
    # timed
    t0 = time.perf_counter()
    for _ in range(n_runs):
        model.predict(source=dummy, verbose=False)
    elapsed = time.perf_counter() - t0
    ms_per_frame = elapsed / n_runs * 1000
    fps = n_runs / elapsed
    return ms_per_frame, fps

for label, weights in [('COCO baseline', COCO_PT), ('DrishtiYOLO', BEST_PT)]:
    ms, fps = benchmark_latency(weights)
    print(f'{label:30s}  {ms:.2f} ms/frame  ({fps:.1f} FPS on T4)')


In [ ]:
# Cell 6 — Error analysis (worst 20 predictions)
# ─────────────────────────────────────────────────────────────
# Strategy: run DrishtiYOLO on val set, collect per-image mAP,
# then visualise the 20 lowest-scoring images with ground truth
# overlaid for qualitative failure-mode categorisation.

from ultralytics import YOLO
from pathlib import Path
import shutil

model = YOLO(BEST_PT)
val_img_dir = Path('/content/idd_yolo/images/val')
val_images  = sorted(val_img_dir.glob('*.jpg'))[:500]  # sample to save time

scores = []
for img in val_images:
    r = model.val(data=YAML_PATH, imgsz=640, batch=1, verbose=False)
    scores.append((r.box.map50, img))

scores.sort(key=lambda x: x[0])
worst_20 = scores[:20]

err_dir = Path('/content/drive/MyDrive/drishti_runs/error_analysis')
err_dir.mkdir(parents=True, exist_ok=True)

for i, (score, img_path) in enumerate(worst_20):
    result = model(img_path, verbose=False)[0]
    ann = result.plot()
    import cv2
    cv2.imwrite(str(err_dir / f'worst_{i+1:02d}_ap{score:.3f}_{img_path.stem}.jpg'), ann)

print(f'Saved 20 worst predictions to {err_dir}')
print('\nManually categorise each into:')
print('  [OCC] Occlusion  [SMALL] Small object  [POSE] Novel pose')
print('  [LIGHT] Lighting  [BLUR] Motion blur    [FP] False positive')
